# Advanced Problems with Solutions: Consuming Iterators Manually

This notebook focuses on manual iterator consumption with `iter()`, `next()`, `StopIteration`, sentinel defaults, header skipping, CSV-style parsing, schema-driven casting, one-shot streams, lookahead, batching, and stream algorithms.

The goal is not just to use `for` loops, but to understand when direct calls to `next()` are useful and how to write reliable, lazy, testable iterator code.

## Best-practice themes

- Use `iter(iterable)` once when you need manual control over consumption.
- Use `next(iterator)` when exhaustion should be exceptional.
- Use `next(iterator, default)` when exhaustion is expected and should not raise.
- Do not catch `StopIteration` too broadly.
- Keep stream parsers lazy unless you explicitly need a list.
- Be clear about whether an object is reusable or one-shot.
- Add assertions for edge cases: empty input, short input, malformed rows, and repeated iteration.


In [1]:
from __future__ import annotations

from collections import Counter, deque, namedtuple
from dataclasses import dataclass
from heapq import heappop, heappush
from itertools import count, islice
from typing import Any, Callable, Iterable, Iterator, TypeVar
import io

T = TypeVar("T")
U = TypeVar("U")


def consume(iterator: Iterator[T]) -> list[T]:
    """Consume an iterator into a list. Useful in tests."""
    return list(iterator)


def assert_raises(expected_exception: type[BaseException], function: Callable, *args, **kwargs) -> BaseException:
    """Tiny helper so every solution can test errors without a framework."""
    try:
        function(*args, **kwargs)
    except expected_exception as exc:
        return exc
    raise AssertionError(f"Expected {expected_exception.__name__}")


## Problem 1 — Manual character reader

Write `read_chars(text, n)` that manually consumes a string iterator using `next()`.

Requirements:

- Return a list of at most `n` characters.
- Stop early when the string is exhausted.
- Do not use a `for` loop, slicing, or list comprehension.
- Raise `ValueError` when `n < 0`.

This is a small exercise, but it forces you to control `StopIteration` directly.


In [2]:
# Solution 1

def read_chars(text: str, n: int) -> list[str]:
    if n < 0:
        raise ValueError("n must be non-negative")

    iterator = iter(text)
    result: list[str] = []

    while len(result) < n:
        try:
            result.append(next(iterator))
        except StopIteration:
            break

    return result


assert read_chars("Python", 3) == ["P", "y", "t"]
assert read_chars("Py", 10) == ["P", "y"]
assert read_chars("", 5) == []
assert read_chars("abc", 0) == []
assert_raises(ValueError, read_chars, "abc", -1)

print("read_chars passed all tests.")


read_chars passed all tests.


## Problem 2 — `next()` with a default sentinel

Write `first_or_default(iterable, default=None, predicate=None)`.

Requirements:

- Return the first item when no predicate is supplied.
- Return the first item satisfying `predicate` when a predicate is supplied.
- Return `default` if no item is available.
- Use `next(iterator, sentinel)` rather than catching `StopIteration` for the empty case.
- It must work when the actual data contains `None`.

Best practice: use a unique sentinel object instead of `None` when `None` is a valid item.


In [3]:
# Solution 2

_SENTINEL = object()


def first_or_default(
    iterable: Iterable[T],
    default: Any = None,
    predicate: Callable[[T], bool] | None = None,
) -> T | Any:
    if predicate is None:
        value = next(iter(iterable), _SENTINEL)
        return default if value is _SENTINEL else value

    matches = (item for item in iterable if predicate(item))
    value = next(matches, _SENTINEL)
    return default if value is _SENTINEL else value


assert first_or_default([10, 20, 30]) == 10
assert first_or_default([], default="missing") == "missing"
assert first_or_default([1, 3, 8, 10], predicate=lambda x: x % 2 == 0) == 8
assert first_or_default([1, 3, 5], default="none", predicate=lambda x: x % 2 == 0) == "none"
assert first_or_default([None, 1, 2], default="missing") is None

print("first_or_default handles empty inputs and None values correctly.")


first_or_default handles empty inputs and None values correctly.


## Problem 3 — Skip exactly N header lines

Write `skip_exactly(iterable, n)` that returns an iterator positioned after `n` consumed items.

Requirements:

- Manually call `next()` exactly `n` times.
- If the input has fewer than `n` items, raise `ValueError` with a useful message.
- Return the same iterator object after skipping, not a list.
- Do not consume more than necessary.

This mirrors CSV files where the first one or two lines contain metadata.


In [4]:
# Solution 3

def skip_exactly(iterable: Iterable[T], n: int) -> Iterator[T]:
    if n < 0:
        raise ValueError("n must be non-negative")

    iterator = iter(iterable)

    for skipped in range(n):
        try:
            next(iterator)
        except StopIteration as exc:
            raise ValueError(f"Expected at least {n} items, but only found {skipped}") from exc

    return iterator


assert list(skip_exactly(["header", "row1", "row2"], 1)) == ["row1", "row2"]
assert list(skip_exactly([1, 2, 3], 0)) == [1, 2, 3]
assert_raises(ValueError, skip_exactly, ["only one"], 2)

source = iter(["a", "b", "c", "d"])
rest = skip_exactly(source, 2)
assert rest is source
assert next(rest) == "c"
assert next(rest) == "d"
assert_raises(StopIteration, next, rest)

print("skip_exactly skips headers without over-consuming.")


skip_exactly skips headers without over-consuming.


## Problem 4 — Pull a schema from the first two CSV lines

A semicolon-delimited data stream starts with:

1. a header row,
2. a type row,
3. data rows.

Build `read_schema(lines)` that manually consumes the first two lines and returns:

- a `namedtuple` class based on the headers,
- a list of converter functions based on the type row,
- the remaining iterator positioned at the first data row.

Supported types:

- `STRING` → `str`
- `CAT` → `str`
- `INT` → `int`
- `DOUBLE` → `float`

Raise clear errors for missing schema lines or unknown types.


In [5]:
# Shared sample data for several problems

CARS_CSV = """Car;MPG;Cylinders;Displacement;Horsepower;Weight;Acceleration;Model;Origin
STRING;DOUBLE;INT;DOUBLE;DOUBLE;DOUBLE;DOUBLE;INT;CAT
Chevrolet Chevelle Malibu;18.0;8;307.0;130.0;3504.;12.0;70;US
Buick Skylark 320;15.0;8;350.0;165.0;3693.;11.5;70;US
Toyota Corolla Mark ii;24.0;4;113.0;95.00;2372.;15.0;70;Japan
Volkswagen 1131 Deluxe Sedan;26.0;4;97.00;46.00;1835.;20.5;70;Europe
Ford Pinto;25.0;4;98.00;0;2046.;19.0;71;US
Datsun 1200;35.0;4;72.00;69.00;1613.;18.0;71;Japan
"""

TYPE_CONVERTERS: dict[str, Callable[[str], Any]] = {
    "STRING": str,
    "CAT": str,
    "INT": int,
    "DOUBLE": float,
}


def split_semicolon_line(line: str) -> list[str]:
    return [part.strip() for part in line.strip().split(";")]


In [6]:
# Solution 4

def read_schema(lines: Iterable[str]) -> tuple[type[tuple], list[Callable[[str], Any]], Iterator[str]]:
    iterator = iter(lines)

    header_line = next(iterator, _SENTINEL)
    if header_line is _SENTINEL:
        raise ValueError("Missing header line")

    type_line = next(iterator, _SENTINEL)
    if type_line is _SENTINEL:
        raise ValueError("Missing type line")

    headers = split_semicolon_line(header_line)
    type_names = split_semicolon_line(type_line)

    if len(headers) != len(type_names):
        raise ValueError(f"Header count {len(headers)} does not match type count {len(type_names)}")

    converters: list[Callable[[str], Any]] = []
    for type_name in type_names:
        try:
            converters.append(TYPE_CONVERTERS[type_name])
        except KeyError as exc:
            raise ValueError(f"Unknown type name: {type_name!r}") from exc

    Row = namedtuple("Car", headers)
    return Row, converters, iterator


Row, converters, remaining = read_schema(io.StringIO(CARS_CSV))
assert Row.__name__ == "Car"
assert Row._fields == ("Car", "MPG", "Cylinders", "Displacement", "Horsepower", "Weight", "Acceleration", "Model", "Origin")
assert converters[1] is float
assert converters[2] is int
assert next(remaining).startswith("Chevrolet Chevelle Malibu")

assert_raises(ValueError, read_schema, [])
assert_raises(ValueError, read_schema, ["A;B"])
assert_raises(ValueError, read_schema, ["A", "UNKNOWN"])

print("read_schema manually consumed metadata and returned the remaining stream.")


read_schema manually consumed metadata and returned the remaining stream.


## Problem 5 — Parse typed rows lazily

Using `read_schema`, implement `parse_typed_rows(lines)`.

Requirements:

- Consume the schema lines with `next()`.
- Lazily yield namedtuple rows.
- Convert each field using the schema converters.
- Skip blank lines.
- Add line numbers to error messages.
- Reject rows with the wrong number of fields.

This is the central CSV-style exercise from this topic, but written as a reusable function with stronger validation.


In [7]:
# Solution 5

def parse_typed_rows(lines: Iterable[str]) -> Iterator[tuple]:
    Row, converters, iterator = read_schema(lines)

    # The first data row is line 3 because lines 1 and 2 are schema lines.
    for line_number, raw_line in enumerate(iterator, start=3):
        line = raw_line.strip()
        if not line:
            continue

        fields = split_semicolon_line(line)
        if len(fields) != len(converters):
            raise ValueError(
                f"Line {line_number}: expected {len(converters)} fields, got {len(fields)}"
            )

        try:
            converted = [converter(value) for converter, value in zip(converters, fields)]
        except ValueError as exc:
            raise ValueError(f"Line {line_number}: could not convert row {fields!r}") from exc

        yield Row(*converted)


cars = list(parse_typed_rows(io.StringIO(CARS_CSV)))

assert len(cars) == 6
assert cars[0].Car == "Chevrolet Chevelle Malibu"
assert cars[0].MPG == 18.0
assert cars[0].Cylinders == 8
assert cars[2].Origin == "Japan"

bad_csv = """A;B
INT;DOUBLE
1;2.5
2;not-a-float
"""
exc = assert_raises(ValueError, lambda: list(parse_typed_rows(io.StringIO(bad_csv))))
assert "Line 4" in str(exc)

print("parse_typed_rows lazily parses and validates typed rows.")


parse_typed_rows lazily parses and validates typed rows.


## Problem 6 — Demonstrate laziness with a noisy stream

Create `noisy_lines(text)` that prints each line as it is pulled, then use `parse_typed_rows` with `next()` to prove rows are read lazily.

Requirements:

- Calling `parse_typed_rows(...)` should not print anything immediately.
- The first `next()` should pull only the schema lines and the first data row.
- The second `next()` should pull only the second data row.

This problem is mostly observational, but it is important for understanding stream parsers.


In [8]:
# Solution 6

def noisy_lines(text: str) -> Iterator[str]:
    for line_number, line in enumerate(text.splitlines(True), start=1):
        print(f"pulling line {line_number}")
        yield line


row_iterator = parse_typed_rows(noisy_lines(CARS_CSV))
print("Iterator created. No lines should have been pulled before this message.")

first_car = next(row_iterator)
second_car = next(row_iterator)

assert first_car.Car == "Chevrolet Chevelle Malibu"
assert second_car.Car == "Buick Skylark 320"

print("Only the needed lines were pulled for two rows.")


Iterator created. No lines should have been pulled before this message.
pulling line 1
pulling line 2
pulling line 3
pulling line 4
Only the needed lines were pulled for two rows.


## Problem 7 — Safe `next()` helper with context

Write `next_required(iterator, message)`.

Requirements:

- Return `next(iterator)` when an item exists.
- Raise `ValueError(message)` instead of `StopIteration` when the iterator is exhausted.
- Preserve exception chaining using `from exc`.

Then rewrite a simpler `read_two_header_lines(lines)` using this helper.


In [9]:
# Solution 7

def next_required(iterator: Iterator[T], message: str) -> T:
    try:
        return next(iterator)
    except StopIteration as exc:
        raise ValueError(message) from exc


def read_two_header_lines(lines: Iterable[str]) -> tuple[str, str, Iterator[str]]:
    iterator = iter(lines)
    header = next_required(iterator, "Missing header line")
    type_row = next_required(iterator, "Missing type line")
    return header, type_row, iterator


header, type_row, rest = read_two_header_lines(["H1;H2", "INT;STRING", "1;x"])
assert header == "H1;H2"
assert type_row == "INT;STRING"
assert list(rest) == ["1;x"]

assert_raises(ValueError, read_two_header_lines, [])
assert_raises(ValueError, read_two_header_lines, ["only header"])

print("next_required converts low-level exhaustion into domain errors.")


next_required converts low-level exhaustion into domain errors.


## Problem 8 — Read until a stop marker

Write `read_until(iterable, stop_value, *, include_stop=False)`.

Requirements:

- Manually consume the iterator with `next()` in a `while` loop.
- Yield items before `stop_value`.
- If `include_stop=True`, include the stop value in the output.
- If the stop value never appears, yield all items.
- Do not use `for`.

This pattern is common in stream protocols and simple parsers.


In [10]:
# Solution 8

def read_until(iterable: Iterable[T], stop_value: T, *, include_stop: bool = False) -> Iterator[T]:
    iterator = iter(iterable)

    while True:
        try:
            item = next(iterator)
        except StopIteration:
            return

        if item == stop_value:
            if include_stop:
                yield item
            return

        yield item


assert list(read_until([1, 2, 3, 4], 3)) == [1, 2]
assert list(read_until([1, 2, 3, 4], 3, include_stop=True)) == [1, 2, 3]
assert list(read_until([1, 2], 99)) == [1, 2]
assert list(read_until([], 99)) == []

print("read_until handles found, missing, and empty stop cases.")


read_until handles found, missing, and empty stop cases.


## Problem 9 — Lookahead parser for comments and blank lines

Create `clean_data_lines(lines)`.

Input rules:

- Blank lines should be skipped.
- Lines starting with `#` are comments and should be skipped.
- A line containing exactly `END` stops the stream.
- Yield all other stripped lines.

Requirement: implement this manually using `next(iterator, sentinel)`.


In [11]:
# Solution 9

def clean_data_lines(lines: Iterable[str]) -> Iterator[str]:
    iterator = iter(lines)

    while True:
        raw_line = next(iterator, _SENTINEL)
        if raw_line is _SENTINEL:
            return

        line = raw_line.strip()

        if not line:
            continue
        if line.startswith("#"):
            continue
        if line == "END":
            return

        yield line


raw = [
    "# file generated by system",
    "",
    "alpha",
    " beta ",
    "# ignore me",
    "gamma",
    "END",
    "delta should not be read",
]

assert list(clean_data_lines(raw)) == ["alpha", "beta", "gamma"]
assert list(clean_data_lines([])) == []
assert list(clean_data_lines(["END", "x"])) == []

print("clean_data_lines uses manual next() with a sentinel default.")


clean_data_lines uses manual next() with a sentinel default.


## Problem 10 — `nth` item without materializing

Write `nth(iterable, index, default=_SENTINEL)`.

Requirements:

- Return the item at zero-based position `index`.
- Do not build a list.
- If the iterable is too short:
  - raise `IndexError` when no default is supplied,
  - otherwise return the supplied default.
- Reject negative indexes with `IndexError`.

Use this to read the 1000th item from an infinite counter.


In [12]:
# Solution 10

def nth(iterable: Iterable[T], index: int, default: Any = _SENTINEL) -> T | Any:
    if index < 0:
        raise IndexError("index must be non-negative")

    iterator = iter(iterable)

    for _ in range(index):
        skipped = next(iterator, _SENTINEL)
        if skipped is _SENTINEL:
            if default is _SENTINEL:
                raise IndexError("iterable is too short")
            return default

    value = next(iterator, _SENTINEL)
    if value is _SENTINEL:
        if default is _SENTINEL:
            raise IndexError("iterable is too short")
        return default
    return value


assert nth(["a", "b", "c"], 0) == "a"
assert nth(["a", "b", "c"], 2) == "c"
assert nth(["a"], 10, default="missing") == "missing"
assert nth(count(100), 1000) == 1100
assert_raises(IndexError, nth, [1, 2], 5)
assert_raises(IndexError, nth, [1, 2], -1)

print("nth finds positions lazily, including in infinite streams.")


nth finds positions lazily, including in infinite streams.


## Problem 11 — Pairwise values using manual priming

Implement `pairwise_manual(iterable)`.

Example:

```python
list(pairwise_manual([10, 20, 30, 40]))
# [(10, 20), (20, 30), (30, 40)]
```

Requirements:

- Manually prime the iterator with the first item.
- If the iterable has fewer than two items, yield nothing.
- Do not use `itertools.pairwise`.


In [13]:
# Solution 11

def pairwise_manual(iterable: Iterable[T]) -> Iterator[tuple[T, T]]:
    iterator = iter(iterable)
    previous = next(iterator, _SENTINEL)

    if previous is _SENTINEL:
        return

    while True:
        current = next(iterator, _SENTINEL)
        if current is _SENTINEL:
            return

        yield previous, current
        previous = current


assert list(pairwise_manual([10, 20, 30, 40])) == [(10, 20), (20, 30), (30, 40)]
assert list(pairwise_manual([10])) == []
assert list(pairwise_manual([])) == []
assert list(islice(pairwise_manual(count(1)), 4)) == [(1, 2), (2, 3), (3, 4), (4, 5)]

print("pairwise_manual uses explicit priming with next().")


pairwise_manual uses explicit priming with next().


## Problem 12 — Batch records from a stream with validation

Write `read_batches(iterable, size, *, strict=False)`.

Requirements:

- Consume the iterable lazily.
- Yield tuples of length `size`.
- If the final batch is incomplete:
  - yield it when `strict=False`,
  - raise `ValueError` when `strict=True`.
- Reject `size <= 0`.
- Use `next(iterator, sentinel)` internally.


In [14]:
# Solution 12

def read_batches(iterable: Iterable[T], size: int, *, strict: bool = False) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)

    while True:
        batch: list[T] = []

        for _ in range(size):
            item = next(iterator, _SENTINEL)
            if item is _SENTINEL:
                if batch and strict:
                    raise ValueError(f"Incomplete final batch of size {len(batch)}; expected {size}")
                if batch:
                    yield tuple(batch)
                return

            batch.append(item)

        yield tuple(batch)


assert list(read_batches([1, 2, 3, 4, 5], 2)) == [(1, 2), (3, 4), (5,)]
assert list(read_batches([1, 2, 3, 4], 2, strict=True)) == [(1, 2), (3, 4)]
assert list(islice(read_batches(count(1), 3), 3)) == [(1, 2, 3), (4, 5, 6), (7, 8, 9)]
assert_raises(ValueError, lambda: list(read_batches([1, 2, 3], 2, strict=True)))
assert_raises(ValueError, lambda: list(read_batches([1, 2], 0)))

print("read_batches works lazily and validates strict mode.")


read_batches works lazily and validates strict mode.


## Problem 13 — Peekable iterator for stream parsers

Implement a `Peekable` iterator.

Requirements:

- `peek()` returns the next value without consuming it.
- `next(peekable)` consumes the value.
- `peek(default)` returns `default` instead of raising if exhausted.
- It must correctly handle real `None` values.
- `has_next()` returns whether another value exists.

This is useful when parsing streams where you need to inspect the next token before deciding what to do.


In [15]:
# Solution 13

_CACHE_EMPTY = object()
_NO_DEFAULT = object()


class Peekable(Iterator[T]):
    def __init__(self, iterable: Iterable[T]):
        self._iterator = iter(iterable)
        self._cache: Any = _CACHE_EMPTY

    def __iter__(self) -> "Peekable[T]":
        return self

    def _ensure_cache(self) -> None:
        if self._cache is _CACHE_EMPTY:
            self._cache = next(self._iterator)

    def peek(self, default: Any = _NO_DEFAULT) -> T | Any:
        try:
            self._ensure_cache()
        except StopIteration:
            if default is _NO_DEFAULT:
                raise
            return default
        return self._cache

    def has_next(self) -> bool:
        try:
            self._ensure_cache()
        except StopIteration:
            return False
        return True

    def __next__(self) -> T:
        if self._cache is not _CACHE_EMPTY:
            value = self._cache
            self._cache = _CACHE_EMPTY
            return value
        return next(self._iterator)


p = Peekable([None, "A", "B"])
assert p.has_next() is True
assert p.peek() is None
assert p.peek() is None
assert next(p) is None
assert p.peek() == "A"
assert next(p) == "A"
assert next(p) == "B"
assert p.peek("done") == "done"
assert p.has_next() is False
assert_raises(StopIteration, p.peek)

print("Peekable supports lookahead without confusing None with empty cache.")


Peekable supports lookahead without confusing None with empty cache.


## Problem 14 — Parse grouped sections with manual lookahead

Given lines like this:

```text
[Europe]
Volkswagen
BMW
[Japan]
Toyota
Honda
```

Write `parse_sections(lines)` that returns a dictionary mapping section names to item lists.

Requirements:

- Use `Peekable` to look ahead for the next section header.
- Ignore blank lines and comments.
- A section header starts with `[` and ends with `]`.
- Items before any section should raise `ValueError`.
- Duplicate section names should merge items.


In [16]:
# Solution 14

def parse_sections(lines: Iterable[str]) -> dict[str, list[str]]:
    cleaned = Peekable(clean_data_lines(lines))
    sections: dict[str, list[str]] = {}
    current_section: str | None = None

    while cleaned.has_next():
        line = next(cleaned)

        if line.startswith("[") and line.endswith("]"):
            current_section = line[1:-1].strip()
            if not current_section:
                raise ValueError("Empty section name")
            sections.setdefault(current_section, [])
            continue

        if current_section is None:
            raise ValueError(f"Item outside a section: {line!r}")

        sections[current_section].append(line)

    return sections


section_text = """
# Cars by origin
[Europe]
Volkswagen
BMW

[Japan]
Toyota
Honda
[Europe]
Audi
END
ignored
"""

assert parse_sections(section_text.splitlines()) == {
    "Europe": ["Volkswagen", "BMW", "Audi"],
    "Japan": ["Toyota", "Honda"],
}

assert_raises(ValueError, parse_sections, ["orphan item"])
assert_raises(ValueError, parse_sections, ["[]"])

print("parse_sections uses lookahead-ready cleaned streams.")


parse_sections uses lookahead-ready cleaned streams.


## Problem 15 — Streaming summary from parsed car rows

Using `parse_typed_rows`, write `summarize_by_origin(lines)`.

Requirements:

- Do not materialize all cars into a list.
- Return a dictionary like:

```python
{
    "US": {"count": 3, "avg_mpg": 19.33},
    ...
}
```

- Treat `MPG == 0` as missing and exclude it from the average, but still count the row.
- Round averages to two decimals.

This is a realistic example of consuming a parsed iterator once to compute aggregate statistics.


In [17]:
# Solution 15

def summarize_by_origin(lines: Iterable[str]) -> dict[str, dict[str, float | int | None]]:
    counts: Counter[str] = Counter()
    mpg_totals: Counter[str] = Counter()
    mpg_counts: Counter[str] = Counter()

    for car in parse_typed_rows(lines):
        counts[car.Origin] += 1

        if car.MPG != 0:
            mpg_totals[car.Origin] += car.MPG
            mpg_counts[car.Origin] += 1

    summary: dict[str, dict[str, float | int | None]] = {}
    for origin in sorted(counts):
        if mpg_counts[origin]:
            avg_mpg: float | None = round(mpg_totals[origin] / mpg_counts[origin], 2)
        else:
            avg_mpg = None

        summary[origin] = {
            "count": counts[origin],
            "avg_mpg": avg_mpg,
        }

    return summary


summary = summarize_by_origin(io.StringIO(CARS_CSV))
assert summary == {
    "Europe": {"count": 1, "avg_mpg": 26.0},
    "Japan": {"count": 2, "avg_mpg": 29.5},
    "US": {"count": 3, "avg_mpg": 19.33},
}

print(summary)


{'Europe': {'count': 1, 'avg_mpg': 26.0}, 'Japan': {'count': 2, 'avg_mpg': 29.5}, 'US': {'count': 3, 'avg_mpg': 19.33}}


## Problem 16 — Merge multiple sorted streams manually

Implement `merge_sorted(*iterables, key=lambda x: x)`.

Requirements:

- Inputs are individually sorted.
- Output should be globally sorted.
- Consume lazily: do not convert inputs to lists.
- Use `next()` to prime each input iterator.
- Use a heap so the function scales to many streams.
- Preserve stable order between streams when keys are equal.


In [18]:
# Solution 16

def merge_sorted(*iterables: Iterable[T], key: Callable[[T], Any] = lambda x: x) -> Iterator[T]:
    heap: list[tuple[Any, int, T, Iterator[T]]] = []

    for stream_index, iterable in enumerate(iterables):
        iterator = iter(iterable)
        first = next(iterator, _SENTINEL)
        if first is not _SENTINEL:
            heappush(heap, (key(first), stream_index, first, iterator))

    while heap:
        _, stream_index, value, iterator = heappop(heap)
        yield value

        next_value = next(iterator, _SENTINEL)
        if next_value is not _SENTINEL:
            heappush(heap, (key(next_value), stream_index, next_value, iterator))


assert list(merge_sorted([1, 4, 7], [2, 5], [3, 6, 8])) == [1, 2, 3, 4, 5, 6, 7, 8]
assert list(merge_sorted([], [1, 2], [])) == [1, 2]

words_a = ["a", "bbb", "cccc"]
words_b = ["dd", "eee", "fffff"]
assert list(merge_sorted(words_a, words_b, key=len)) == ["a", "dd", "bbb", "eee", "cccc", "fffff"]

infinite_merge = merge_sorted(count(0, 2), count(1, 2))
assert list(islice(infinite_merge, 10)) == list(range(10))

print("merge_sorted lazily merges finite and infinite sorted streams.")


merge_sorted lazily merges finite and infinite sorted streams.


## Problem 17 — Stream parser mini-project: typed table reader

Create a reusable `TypedTableReader` class.

Requirements:

- Accept a text string or any iterable of lines.
- Use the same two-line schema format as the car data.
- Iteration should parse rows lazily.
- If constructed from a text string, it should be reusable.
- If constructed from a one-shot iterator, it will be one-shot; document this with a test.
- Provide `.first(default=None)` that manually returns the first parsed row or a default.


In [19]:
# Solution 17

class TypedTableReader:
    def __init__(self, source: str | Iterable[str]):
        self._source = source

    def _line_iterable(self) -> Iterable[str]:
        if isinstance(self._source, str):
            return io.StringIO(self._source)
        return self._source

    def __iter__(self) -> Iterator[tuple]:
        return parse_typed_rows(self._line_iterable())

    def first(self, default: Any = None) -> Any:
        return next(iter(self), default)


reader = TypedTableReader(CARS_CSV)
assert reader.first().Car == "Chevrolet Chevelle Malibu"
assert len(list(reader)) == 6
assert len(list(reader)) == 6  # reusable because source is a string

one_shot_reader = TypedTableReader(iter(CARS_CSV.splitlines(True)))
assert len(list(one_shot_reader)) == 6
assert_raises(ValueError, lambda: list(one_shot_reader))  # source iterator was consumed, so schema is missing

empty_table = """A;B
INT;STRING
"""
assert TypedTableReader(empty_table).first(default="no rows") == "no rows"

print("TypedTableReader wraps schema parsing in a convenient iterable object.")


TypedTableReader wraps schema parsing in a convenient iterable object.


## Problem 18 — Debugging challenge: catching `StopIteration` correctly

The following function is wrong because it catches `StopIteration` around too much code:

```python
def bad_sum_pairs(iterable):
    iterator = iter(iterable)
    total = 0
    while True:
        try:
            a = next(iterator)
            b = next(iterator)
            total += a / b
        except StopIteration:
            return total
```

If `total += a / b` calls user code that accidentally raises `StopIteration`, the function would silently stop.

Write `safe_sum_pairs(iterable)`.

Requirements:

- Read items two at a time.
- Add `a / b` to a total.
- Ignore a trailing unpaired item.
- Catch `StopIteration` only around `next()` calls.
- Let other errors, like `ZeroDivisionError`, propagate.


In [20]:
# Solution 18

def safe_sum_pairs(iterable: Iterable[float]) -> float:
    iterator = iter(iterable)
    total = 0.0

    while True:
        try:
            a = next(iterator)
        except StopIteration:
            return total

        try:
            b = next(iterator)
        except StopIteration:
            return total

        total += a / b


assert safe_sum_pairs([10, 2, 9, 3]) == 8.0
assert safe_sum_pairs([10, 2, 9]) == 5.0
assert safe_sum_pairs([]) == 0.0
assert_raises(ZeroDivisionError, safe_sum_pairs, [1, 0])

print("safe_sum_pairs catches exhaustion narrowly and preserves real errors.")


safe_sum_pairs catches exhaustion narrowly and preserves real errors.


# Final checklist

When manually consuming iterators, ask:

- Do I need `next(iterator)` or would a `for` loop be clearer?
- Should exhaustion raise an exception or return a default?
- Am I catching `StopIteration` only around the `next()` call that can raise it?
- Have I handled empty and short inputs?
- Am I accidentally storing a one-shot iterator where I need a reusable iterable?
- Does my parser consume exactly the header/schema rows and no more?
- Does the code stay lazy for large files or infinite streams?
- Are malformed rows reported with enough context, such as line numbers?
